In [ ]:
from bs4 import BeautifulSoup, Comment
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

import requests
import time
import csv
from io import StringIO

import sys
from pathlib import Path

In [ ]:
pd.set_option('display.max_columns', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
     ROOT = ROOT.parent
src_path = str(ROOT/"src")
if src_path not in sys.path:
    sys.path.insert(0,src_path)

from config import START_YEAR, END_YEAR, BASE_URL, HTTP_HEADER
import utils

### [Basketball-Reference Bot Policy](https://www.sports-reference.com/bot-traffic.html)

Basketball reference enforces a 1 day block if you are caught accessing the site more than 20 times a second.  All cells that run loops which collect the data should have `time.sleep(x)` in the loop where x $\geq3$.  In what follows we use `time.sleep(3.3)` for good measure.  Currently the cell should take about 8 minutes to run.

#### IF CHANGES HAVE BEEN MADE TO `utils.scrape_per_game_tables()` YOU NEED TO RESTART THE KERNEL, UNCOMMENT THIS CELL, AND RUN ALL.  DON'T FORGET TO RE-COMMENT OUT THIS CELL IF YOU HAVE ALREADY DOWNLOADED THE DATA AND NO CHANGES HAVE BEEN MADE TO `utils.scrape_per_game_tables()` OR YOU WILL BE SAD WHEN YOU RERUN THE NOTEBOOK!

In [ ]:
# utils.scrape_per_game_tables(START_YEAR,END_YEAR)

when we import the datafram from the csv it is important to add a comment keyword or it will mislabel the columns using the descriptive comment instead of the names.

Example for the 2025-2026 season

In [ ]:
season = '2025-2026'
df = pd.read_csv(str(ROOT)+f'/data/{season}_season/per_game_stats_{season}.csv',encoding='utf-8',comment='#')

In [ ]:
df.sort_values('MVP', ascending=False)

Note that players who have been traded during the season show up multiple times.  One for each team as well as an aggregate row where the team is listed as "#TM" and # is the number of teams they have been on. We have made the team wins/losses for these rows NaNs.  Here are the people who were on three teams.

In [ ]:
df.loc[df['Team']=='3TM']

Now we import all of the data into a single pandas dataframe

In [ ]:
df_tot = utils.player_stat_dataframe(START_YEAR,END_YEAR)

In [ ]:
df_tot.sample(10)

In [ ]:
df_mvp = df_tot.loc[df_tot['MVP'] != 0.0].reset_index(drop=True)
df_mvp.sample(5)

In [ ]:
features = df_mvp.drop(columns=['Year','Code','Player','Team','Pos','Age','FG%','3P%', '2P%', 'FT%', 'FGA', '3PA', '2PA', 'FTA', 'PF']).columns[:-1]
target = df_mvp.columns[-1]

In [ ]:
for i in range(0,len(features),5):
    sns.pairplot(
        df_tot,
        x_vars=features[i:i+5],
        y_vars=target,
    )
    plt.show()

In [ ]:
for i in range(0,len(features),5):
    sns.pairplot(
        df_mvp,
        x_vars=features[i:i+5],
        y_vars=target,
    )
    plt.show()

In [ ]:
df_new = df_tot.copy()
df_new['RecievedVotes'] = (df_tot['MVP'] != 0.0).astype('int64')
df_new['TradedMidSeason'] = df_tot['Team'].str.endswith('TM').groupby([df_tot['Year'], df_tot['Code']]).transform('any').astype('int64') #adding a col to show of a player got traded

In [ ]:
custom_palette = {
    0: (0.12, 0.47, 0.71, 0.4),  # Light blue
    1: (1.00, 0.50, 0.05, 0.4)   # Solid orange
}
sns.pairplot(
    df_new,
    x_vars=['W%','PTS','FT','G','TOV'],
    y_vars=['W%','PTS','FT','G','TOV'],
    hue='RecievedVotes',
    plot_kws={"s":10},
    palette=custom_palette
)
plt.show()

In [ ]:
sns.pairplot(
    df_new,
    x_vars=features[0:11],
    y_vars=features[0:11],
    hue='RecievedVotes',
    plot_kws={"alpha": 0.2,"s":10}
)
plt.show()

In [ ]:
sns.pairplot(
    df_new,
    x_vars=features[11:-1],
    y_vars=features[11:-1],
    hue='RecievedVotes',
    plot_kws={"alpha": 0.2,"s":10}
)
plt.show()

In [ ]:
print(features)
X = df_new[features]
y = df_new['RecievedVotes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

In [ ]:
# TODO: drop the features that involve NaNs and run LogisticRegression.
print(features)

X = df_new[features]
y = df_new['RecievedVotes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

In [ ]:
for i in range(0,len(features),5):
    sns.pairplot(
        df_new,
        x_vars=features[i:i+5],
        y_vars=features[i:i+5],
        hue='RecievedVotes',
        plot_kws={"alpha": 0.2,"s":10}
    )
    plt.show()

In [ ]:
#heatmap to see correlations
season_rows=df_new[(df_new['TradedMidSeason']==0) | df_new['Team'].str.endswith('TM')]
rel=season_rows[(season_rows['MP']>=15) & (season_rows['G']>=20)]

corr=rel[list(features) + ['MVP']].corr(method='spearman')
mask=np.triu(np.ones_like(corr,dtype=bool))
plt.figure(figsize=(12,10))
sns.heatmap(corr,mask=mask,cmap='vlag',center=0,vmin=-1,vmax=1,annot=True,fmt='.2f',annot_kws={'size':7})
plt.title('spearman corr--1 row per player-season for MP>=15')
plt.show()

In [ ]:
#clustering groups features whose rows in heatmap look alike, meaning they correlate with evything else in a similar way. So for feature selection we move up the tree from the leaves and when we hit a group we keep on representative
sns.clustermap(corr,cmap='vlag',center=0,vmin=-1,vmax=1,figsize=(12,12))

the heatmap and clustermap give some sense of stars vs bench players and a big play time effect.

In [ ]:
#addressing my question of finding out among players good enough to get votes, what separates the winner from the rest
vg=rel[rel['MVP']>0]
corr_vg=vg[list(features)+['MVP']].corr(method='spearman')
mask=np.triu(np.ones_like(corr_vg,dtype=bool))
plt.figure(figsize=(12,10))
sns.heatmap(corr_vg,mask=mask,cmap='vlag',center=0,vmin=-1,vmax=1,annot=True,fmt='.2f',annot_kws={'size':7})
plt.title(f'spearman corr vote getters only (n={len(vg)})')
plt.show()

this heatmap give a sense of what separates MVP players from each other

In [ ]:
#creating reduced feature set based on heatmaps to get a baseline model working

feats=['PTS','W%','FT','eFG%','TRB','AST','BLK','STL']
data=rel.reset_index(drop=True)

X,y,years = data[feats],data['MVP'],data['Year']
data['pred']=0.0

for tr,te in LeaveOneGroupOut().split(X,y,groups=years):
    model=HistGradientBoostingRegressor(random_state=12).fit(X.iloc[tr], y.iloc[tr]) #HGBR handles NaNs
    data.loc[te,'pred']=model.predict(X.iloc[te])


In [ ]:
def evaluate(df,score_col):
    hits,ranks = 0,[]
    for yr, g in df.groupby('Year'):
        actual=g.loc[g['MVP'].idxmax(), 'Code']
        order=g.sort_values(score_col, ascending=False)['Code'].tolist()
        hits += order[0]==actual
        ranks.append(order.index(actual) + 1)
    print(f"{score_col}: picked the MVP in {hits}/{len(ranks)} seasons, "f"real winner's average predicted rank = {np.mean(ranks):.1f}")

evaluate(data, 'pred')
evaluate(data, 'PTS') # naive baseline: 'league's top scorer wins'

In [ ]:
#checking what the model didn't capture
rows=[]
for yr,g in data.groupby('Year'):
    g=g.sort_values('pred',ascending=False)
    actual=g.loc[g['MVP'].idxmax()]
    if g.iloc[0]['Code']!=actual['Code']:
        rows.append({'Year': yr, 'Predicted': g.iloc[0]['Player'], 'Pred W%': g.iloc[0]['W%'], 'Pred PTS': g.iloc[0]['PTS'],
                     'Actual': actual['Player'], 'Actual W%': actual['W%'], 'Actual PTS': actual['PTS'],
                     'Actual rank': g['Code'].tolist().index(actual['Code'])+ 1})
pd.DataFrame(rows)

In [ ]:
#which features is the mmodel actually using?
model=HistGradientBoostingRegressor(random_state=0).fit(X, y)
imp=permutation_importance(model, X, y, n_repeats=10, random_state=12)
pd.Series(imp.importances_mean, index=feats).sort_values().plot.barh()
plt.title('Permutation importance')
plt.show()